In [3]:
"""
Phase 9 · Animated World Map (2022 Q1 → 2026 Q1, per-quarter)

Paste the entire contents of this file into a new notebook cell.
Requires the following variables already in kernel scope:
    df, country_wide, region_wide, countries_geojson, GEOJSON_ISO3_KEY,
    REGION_CENTROIDS, CATEGORY_COLORS, MAPBOX_TOKEN, MAPBOX_STYLE, OUTPUTS_DIR
"""

import io
import warnings
import numpy as np
import plotly.graph_objects as go
from PIL import Image
import re
import json
import math
import requests
import pandas as pd
from pathlib import Path

import plotly.graph_objects as go

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    module=r"plotly\..*",
)
warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    message=r".*(Scattermapbox|Choroplethmapbox|scattermapbox|choroplethmapbox).*",
)


In [4]:
# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT          = Path('..').resolve()
DATA_DIR      = ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
CONFIG_DIR    = DATA_DIR / 'config'
OUTPUTS_DIR   = ROOT / 'outputs'
OUTPUTS_DIR.mkdir(exist_ok=True)

WORKBOOK_PATH     = ROOT / 'DREF_MasterDataset_v1.1 .xlsx'
COUNTRY_API_CACHE = PROCESSED_DIR / 'country_api_reference_raw.json'
COUNTRY_API_URL   = 'https://goadmin.ifrc.org/api/v2/country/'

# ── Mapbox config ─────────────────────────────────────────────────────────────
mapbox_cfg   = json.loads((CONFIG_DIR / 'mapbox.local.json').read_text(encoding='utf-8'))
MAPBOX_TOKEN = mapbox_cfg['mapbox_token']
MAPBOX_STYLE = mapbox_cfg['style_url']

print(f"Workbook    : {WORKBOOK_PATH.name}")
print(f"Mapbox style: {MAPBOX_STYLE}")

Workbook    : DREF_MasterDataset_v1.1 .xlsx
Mapbox style: mapbox://styles/go-ifrc/ckrfe16ru4c8718phmckdfjh0


In [5]:
def slugify_column(name: str) -> str:
    """Convert a column header to lowercase_with_underscores."""
    text = re.sub(r'[^0-9a-zA-Z]+', '_', str(name).strip().lower())
    return re.sub(r'_+', '_', text).strip('_')

raw = pd.read_excel(WORKBOOK_PATH, sheet_name='ALL_DATA')
df  = raw.rename(columns={c: slugify_column(c) for c in raw.columns}).copy()

print(f"Loaded {len(df):,} rows × {len(df.columns)} columns from ALL_DATA")
df.head(2)

Loaded 2,506 rows × 78 columns from ALL_DATA


c:\Users\arun.gandhi\Downloads\DREF_GA_visualizations\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,index,appeal_id,code_number,child_id,appeal_s_allocation,pillar,appeal_type,allocation_type,country,region_code,...,amount_transferred,canada_gvt,echo,nlrc,belgian_gvt,other_donor_chf,donor_pns,reimbursed_for_finance,comments,year
0,1,MDRCD006,006,NaN,NaN,Anticipatory,i-DREF,Grant,Democratic Republic of the Congo,AF,...,NaN,NaN,NaN,67659.0,NaN,NaN,NaN,1083.0,NaN,2009
1,2,MDRTG002,002,NaN,NaN,Response,DREF,Grant,Togo,AF,...,NaN,NaN,NaN,14961.0,NaN,NaN,NaN,4257.0,NaN,2009


In [6]:
# String columns: strip whitespace
for col in ['appeal_type', 'allocation_type', 'country', 'region', 'silent_cancelled']:
    if col in df.columns:
        df[col] = df[col].fillna('').astype(str).str.strip()

# Approval date derived from the ENC start date column (same convention as nb 01 & 05)
df['approval_date']  = pd.to_datetime(df['date_of_approval_enc_start_date'], errors='coerce')
df['approval_year']  = df['approval_date'].dt.year.astype('Int64')
df['approval_month'] = df['approval_date'].dt.month.astype('Int64')

# Total approved CHF → numeric
df['total_approved_chf'] = pd.to_numeric(df['total_approved_chf'], errors='coerce')

missing_dates = df['approval_date'].isna().sum()
print(f"Rows with unparseable approval date (will be excluded): {missing_dates:,}")
print(f"total_approved_chf nulls: {df['total_approved_chf'].isna().sum():,}")
df[['appeal_type', 'allocation_type', 'country', 'region', 'approval_date', 'total_approved_chf']].head(3)

Rows with unparseable approval date (will be excluded): 2
total_approved_chf nulls: 0


,appeal_type,allocation_type,country,region,approval_date,total_approved_chf
0,i-DREF,Grant,Democratic Republic of the Congo,Africa,2009-01-12,146404
1,DREF,Grant,Togo,Africa,2009-01-18,75376
2,DREF,Grant,Central African Republic,Africa,2009-01-18,30100


In [7]:
before = len(df)

# Drop rows without a parseable approval date
df = df[df['approval_date'].notna()].copy()

# Keep 2022-01-01 → 2026-03-31  (2022 through 2026 Q1 inclusive)
df = df[df['approval_year'].between(2022, 2026)].copy()
df = df[~((df['approval_year'] == 2026) & (df['approval_month'] > 3))].copy()

print(f"Rows before date filter : {before:,}")
print(f"Rows after  date filter : {len(df):,}")
print(f"Date range              : {df['approval_date'].min().date()} → {df['approval_date'].max().date()}")

Rows before date filter : 2,506
Rows after  date filter : 893
Date range              : 2022-01-04 → 2026-03-31


In [8]:
n0 = len(df)

# Secretariat entries (Africa Secretariat, America Secretariat, MENA Secretariat)
# are KEPT in df so they appear in regional totals.
# They have no valid iso3 after the API merge, so they are automatically excluded
# from the country-level circles (which filter on iso3.notna()).
# Loans and Silent/Cancelled are also kept — all funding types included.

n1 = n0  # no rows removed at this stage

print(f"All rows retained (no secretariat removal): {n1:,}")
print(f"\nRemaining appeal_type values: {sorted(df['appeal_type'].unique())}")
print(f"Allocation types kept       : {sorted(df['allocation_type'].unique())}")
print(f"\nSecretariat entries still present:")
print(df[df['country'].str.contains('Secretariat', na=False)][['country', 'region', 'total_approved_chf']].head())

All rows retained (no secretariat removal): 893

Remaining appeal_type values: ['DREF', 'EA', 'EAP', 'a-DREF', 'i-DREF', 's-EAP']
Allocation types kept       : ['Grant', 'Loan']

Secretariat entries still present:
                  country    region  total_approved_chf
1713  America Secretariat  Americas               60000
1714  America Secretariat  Americas               60000
1753   Africa Secretariat    Africa              500000
1868   Africa Secretariat    Africa              250000
1957     MENA Secretariat      MENA              250000


In [ ]:
# ── Appeal Type → 3 display categories ───────────────────────────────────────
CATEGORY_MAP = {
    'DREF':   'DREF',
    'i-DREF': 'DREF',
    'a-DREF': 'DREF',
    'EA':     'EA',
    'EAP':    'EAP',
    's-EAP':  'EAP',
}

df['category'] = df['appeal_type'].map(CATEGORY_MAP)

unclassified = df[df['category'].isna()]['appeal_type'].unique()
if len(unclassified):
    print(f"WARNING — dropping unclassified appeal types: {sorted(unclassified)}")
df = df[df['category'].notna()].copy()

# ── Region display constants (all keyed with Asia-Pacific HYPHEN) ─────────────
# Prior notebooks used 'Asia Pacific' (space) which caused invisible circles.
REGION_CENTROIDS = {
    'Africa':       {'longitude':  25.0, 'latitude':  2.0,  'zoom': 3},
    'Americas':     {'longitude': -75.0, 'latitude':  5.0,  'zoom': 3},
    'Asia-Pacific': {'longitude': 118.0, 'latitude': 15.0,  'zoom': 3},  # ← hyphen fixed
    'Europe':       {'longitude':  20.0, 'latitude': 50.0,  'zoom': 3},
    'MENA':         {'longitude':  40.0, 'latitude': 28.0,  'zoom': 4},
}

CATEGORY_COLORS = {
    'DREF': '#B4612D',
    'EA':   '#AD914C',
    'EAP':  '#627A54',
}

print(f"Rows after category classification: {len(df):,}")
print(df.groupby('category')['total_approved_chf']
        .agg(records='count', total_chf='sum')
        .assign(total_chf=lambda x: x['total_chf'].map('CHF {:,.0f}'.format)))

In [ ]:
# Country name overrides: Excel spelling → IFRC GO API name
# (Same mapping validated in notebook 05)
COUNTRY_NAME_OVERRIDES = {
    # Formal names
    'Iran':                             'Iran, Islamic Republic of',
    'Syria':                            'Syrian Arab Republic',
    'Russia':                           'Russian Federation',
    'Tanzania':                         'Tanzania, United Republic of',
    'Gambia':                           'Gambia, Republic of The',
    'Eswatini':                         'Eswatini, Kingdom of',
    'Lao PDR':                          "Lao People's Democratic Republic",
    'Micronesia':                       'Micronesia, Federated States of',
    # Hyphenation / spacing
    'Guinea Bissau':                    'Guinea-Bissau',
    'Timor Leste':                      'Timor-Leste',
    'Republic of Congo':                'Congo',
    # Renamed countries
    'Turkey':                           'Türkiye',
    'Cape Verde':                       'Cabo Verde',
    # Different convention
    'Democratic Republic of the Congo': 'Democratic Republic of Congo',
    'Vietnam':                          'Viet Nam',
    'Czechia':                          'Czech Republic',
}

df['country_clean'] = df['country'].replace(COUNTRY_NAME_OVERRIDES)

# ── Load IFRC GO country reference (cache-first) ──────────────────────────────
def fetch_country_api(base_url, page_size=200):
    rows, next_url, total_expected = [], f"{base_url}?limit={page_size}", None
    session = requests.Session()
    while next_url:
        resp = session.get(next_url, timeout=60)
        resp.raise_for_status()
        payload = resp.json()
        if total_expected is None:
            total_expected = payload.get('count', 0)
        rows.extend(payload.get('results', []))
        next_url = payload.get('next')
        print(f"  Fetched {len(rows)} / {total_expected} records …")
    return rows

if COUNTRY_API_CACHE.exists():
    print(f"Loading country reference from cache …")
    country_rows = json.loads(COUNTRY_API_CACHE.read_text(encoding='utf-8'))
else:
    print("Cache not found — fetching from IFRC GO API …")
    country_rows = fetch_country_api(COUNTRY_API_URL)
    COUNTRY_API_CACHE.write_text(
        json.dumps(country_rows, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    print(f"Cached to {COUNTRY_API_CACHE}")

country_api = pd.DataFrame(country_rows)
country_api = country_api[country_api['record_type'] == 1].copy()
country_api['api_country_name'] = country_api['name'].astype(str).str.strip()
country_api['iso3']      = country_api['iso3'].fillna('').astype(str).str.strip()
country_api['longitude'] = country_api['centroid'].apply(
    lambda v: v.get('coordinates', [None, None])[0] if isinstance(v, dict) else None
)
country_api['latitude']  = country_api['centroid'].apply(
    lambda v: v.get('coordinates', [None, None])[1] if isinstance(v, dict) else None
)
country_lookup = country_api[['api_country_name', 'iso3', 'longitude', 'latitude']].drop_duplicates()

df = df.merge(country_lookup, left_on='country_clean', right_on='api_country_name', how='left')

matched   = df['iso3'].notna() & df['iso3'].ne('')
unmatched_countries = df.loc[~matched, 'country'].unique()

print(f"\nMatched rows : {matched.sum():,} / {len(df):,}")
if len(unmatched_countries):
    print(f"Unmatched countries ({len(unmatched_countries)}): {sorted(unmatched_countries)}")
else:
    print("All countries matched ✓")

In [ ]:
summary = (
    df.groupby(['region', 'category'])['total_approved_chf']
      .sum()
      .reset_index()
      .pivot_table(index='region', columns='category', values='total_approved_chf',
                   aggfunc='sum', fill_value=0)
)
summary['TOTAL'] = summary.sum(axis=1)
summary = summary.sort_values('TOTAL', ascending=False)
summary_fmt = summary.map(lambda x: f"CHF {x:,.0f}")

print("Total funding 2022 – 2026 Q1  |  Region × Category\n")
display(summary_fmt)

print(f"\nGrand total : CHF {df['total_approved_chf'].sum():,.0f}")
print(f"Unique countries with iso3 : {df.loc[df['iso3'].notna(), 'country_clean'].nunique()}")
print(f"Regions in data            : {sorted(df['region'].unique())}")

In [ ]:
# ── Country-level aggregation ─────────────────────────────────────────────────
# Sum CHF per (country, region, iso3, coordinates, category) across all quarters
country_cat = (
    df[df['iso3'].notna() & df['iso3'].ne('')]
      .groupby(
          ['country_clean', 'region', 'iso3', 'longitude', 'latitude', 'category'],
          as_index=False
      )['total_approved_chf']
      .sum()
)

# Pivot categories to columns
country_wide = country_cat.pivot_table(
    index=['country_clean', 'region', 'iso3', 'longitude', 'latitude'],
    columns='category',
    values='total_approved_chf',
    aggfunc='sum',
    fill_value=0
).reset_index()
country_wide.columns.name = None

# Ensure all three category columns exist even if a region has none
for cat in ['DREF', 'EA', 'EAP']:
    if cat not in country_wide.columns:
        country_wide[cat] = 0

country_wide = country_wide.rename(columns={
    'DREF': 'DREF_chf',
    'EA':   'EA_chf',
    'EAP':  'EAP_chf',
})
country_wide['total_chf'] = country_wide['DREF_chf'] + country_wide['EA_chf'] + country_wide['EAP_chf']

# Category fractions (0–1) for circle sector sizing
country_wide['DREF_frac'] = country_wide['DREF_chf'] / country_wide['total_chf'].replace(0, np.nan)
country_wide['EA_frac']   = country_wide['EA_chf']   / country_wide['total_chf'].replace(0, np.nan)
country_wide['EAP_frac']  = country_wide['EAP_chf']  / country_wide['total_chf'].replace(0, np.nan)
country_wide[['DREF_frac', 'EA_frac', 'EAP_frac']] = (
    country_wide[['DREF_frac', 'EA_frac', 'EAP_frac']].fillna(0)
)

print(f"Country-level rows: {len(country_wide):,}")
country_wide.sort_values('total_chf', ascending=False).head(8)[
    ['country_clean', 'region', 'iso3', 'DREF_chf', 'EA_chf', 'EAP_chf', 'total_chf']
]

In [ ]:
# ── Region-level aggregation ──────────────────────────────────────────────────
region_cat = (
    df[df['category'].notna()]
      .groupby(['region', 'category'], as_index=False)['total_approved_chf']
      .sum()
)

region_wide = region_cat.pivot_table(
    index='region',
    columns='category',
    values='total_approved_chf',
    aggfunc='sum',
    fill_value=0
).reset_index()
region_wide.columns.name = None

for cat in ['DREF', 'EA', 'EAP']:
    if cat not in region_wide.columns:
        region_wide[cat] = 0

region_wide = region_wide.rename(columns={
    'DREF': 'DREF_chf',
    'EA':   'EA_chf',
    'EAP':  'EAP_chf',
})
region_wide['total_chf'] = region_wide['DREF_chf'] + region_wide['EA_chf'] + region_wide['EAP_chf']

# Attach centroid coordinates from the lookup dict
region_wide['longitude'] = region_wide['region'].map(
    lambda r: REGION_CENTROIDS.get(r, {}).get('longitude')
)
region_wide['latitude'] = region_wide['region'].map(
    lambda r: REGION_CENTROIDS.get(r, {}).get('latitude')
)

# Category fractions
region_wide['DREF_frac'] = region_wide['DREF_chf'] / region_wide['total_chf'].replace(0, np.nan)
region_wide['EA_frac']   = region_wide['EA_chf']   / region_wide['total_chf'].replace(0, np.nan)
region_wide['EAP_frac']  = region_wide['EAP_chf']  / region_wide['total_chf'].replace(0, np.nan)
region_wide[['DREF_frac', 'EA_frac', 'EAP_frac']] = (
    region_wide[['DREF_frac', 'EA_frac', 'EAP_frac']].fillna(0)
)

print(f"Region-level rows: {len(region_wide):,}")
print()
display(
    region_wide[['region', 'longitude', 'latitude', 'DREF_chf', 'EA_chf', 'EAP_chf', 'total_chf',
                 'DREF_frac', 'EA_frac', 'EAP_frac']]
      .sort_values('total_chf', ascending=False)
      .style.format({
          'DREF_chf': 'CHF {:,.0f}', 'EA_chf': 'CHF {:,.0f}', 'EAP_chf': 'CHF {:,.0f}',
          'total_chf': 'CHF {:,.0f}',
          'DREF_frac': '{:.1%}', 'EA_frac': '{:.1%}', 'EAP_frac': '{:.1%}',
      })
)

In [ ]:
GEOJSON_CACHE = PROCESSED_DIR / 'countries.geojson'
GEOJSON_URL   = (
    'https://raw.githubusercontent.com/datasets/geo-countries/master/data/countries.geojson'
)

if GEOJSON_CACHE.exists():
    print(f"Loading GeoJSON from cache: {GEOJSON_CACHE}")
    countries_geojson = json.loads(GEOJSON_CACHE.read_text(encoding='utf-8'))
else:
    print(f"Downloading GeoJSON from {GEOJSON_URL} …")
    resp = requests.get(GEOJSON_URL, timeout=120)
    resp.raise_for_status()
    countries_geojson = resp.json()
    GEOJSON_CACHE.write_text(
        json.dumps(countries_geojson, ensure_ascii=False), encoding='utf-8'
    )
    print(f"Cached to {GEOJSON_CACHE}")

n_features = len(countries_geojson.get('features', []))
print(f"GeoJSON features: {n_features:,}")

# The geo-countries dataset uses 'ISO3166-1-Alpha-3' (not 'ISO_A3')
GEOJSON_ISO3_KEY = 'ISO3166-1-Alpha-3'
sample_props = countries_geojson['features'][0]['properties']
print(f"Feature property keys : {list(sample_props.keys())}")
print(f"Join key used         : '{GEOJSON_ISO3_KEY}' → e.g. '{sample_props.get(GEOJSON_ISO3_KEY)}'")

In [ ]:
# ── Verify GeoJSON ↔ data iso3 coverage ──────────────────────────────────────
geojson_iso3 = {
    f['properties'].get(GEOJSON_ISO3_KEY, '').strip()
    for f in countries_geojson['features']
}
data_iso3 = set(country_wide['iso3'].dropna().unique())

matched_geo = data_iso3 & geojson_iso3
missing_geo = data_iso3 - geojson_iso3   # countries in data but absent from GeoJSON

print(f"Countries in data          : {len(data_iso3)}")
print(f"Matched in GeoJSON (iso3)  : {len(matched_geo)}")
if missing_geo:
    print(f"Missing from GeoJSON ({len(missing_geo)}): {sorted(missing_geo)}")
else:
    print("All data countries covered by GeoJSON ✓")

In [ ]:


def make_pie_sectors(
    lon: float,
    lat: float,
    total_chf: float,
    fractions: dict,
    colors: dict,
    max_chf: float,
    max_radius_deg: float = 4.0,
    min_radius_deg: float = 0.3,
    n_points: int = 60,
) -> list:
    if total_chf <= 0 or max_chf <= 0:
        return []

    radius_lat = max(max_radius_deg * math.sqrt(total_chf / max_chf), min_radius_deg)
    cos_lat = math.cos(math.radians(lat))
    radius_lon = radius_lat / cos_lat if abs(cos_lat) > 1e-6 else radius_lat

    traces = []
    start_deg = -90.0

    for cat, frac in fractions.items():
        if frac <= 0:
            continue
        sweep_deg = frac * 360.0
        end_deg   = start_deg + sweep_deg
        n_arc = max(2, int(round(n_points * frac)))
        angles_rad = [
            math.radians(start_deg + i * sweep_deg / (n_arc - 1))
            for i in range(n_arc)
        ]
        arc_lons = [lon + radius_lon * math.cos(a) for a in angles_rad]
        arc_lats = [lat + radius_lat * math.sin(a) for a in angles_rad]
        lons = [lon] + arc_lons + [lon]
        lats = [lat] + arc_lats + [lat]

        traces.append(dict(
            type='scattermapbox',       # ← must be scattermapbox (not scattermap)
            lon=lons, lat=lats,
            mode='lines', fill='toself', fillcolor=colors[cat],
            line=dict(color=colors[cat], width=0.5),
            opacity=0.85, hoverinfo='skip', showlegend=False, name=cat,
        ))
        start_deg = end_deg

    return traces

# sanity check
_test_traces = make_pie_sectors(
    lon=0.0, lat=0.0, total_chf=5_000_000,
    fractions={'DREF': 0.6, 'EA': 0.25, 'EAP': 0.15},
    colors=CATEGORY_COLORS, max_chf=50_000_000,
)
print(f"make_pie_sectors sanity check -> {len(_test_traces)} traces (expected 3)")
for t in _test_traces:
    print(f"  name={t['name']:5s}  fillcolor={t['fillcolor']}  points={len(t['lon'])}")

In [ ]:
def build_animated_world_map(
    width=1000,
    height=620,
    max_radius_deg=14.0,
    min_radius_deg=2.5,
    sector_points=24,
):
    """
    Build lightweight quarterly frame data for a world funding animation.

    This version is optimized for GIF export rather than interactive HTML:
    - one static figure per quarter instead of embedded Plotly frames
    - fewer polygon points per pie sector for faster rendering
    - smaller default canvas to reduce file size and render time
    """
    REGIONS_ORDER = ['Africa', 'Americas', 'Asia-Pacific', 'Europe', 'MENA']
    CATS_ORDER    = ['DREF', 'EA', 'EAP']
    N_CIRCLE      = len(REGIONS_ORDER) * len(CATS_ORDER)  # always 15

    # ── Quarter list ──────────────────────────────────────────────────────────
    quarters = [
        (y, q)
        for y in range(2022, 2027)
        for q in range(1, 5)
        if not (y == 2026 and q > 1)
    ]

    # ── Add quarter column ────────────────────────────────────────────────────
    df_q = df.copy()
    df_q['quarter'] = df_q['approval_date'].dt.quarter

    # ── Pre-compute per-quarter aggregations + global max for colour scale ────
    quarterly = []
    max_country_chf = 0.0
    max_region_chf = 0.0

    for year, q in quarters:
        mask = (df_q['approval_year'] == year) & (df_q['quarter'] == q)
        sub  = df_q[mask]

        # Country pivot
        c = (sub[sub['iso3'].notna() & sub['iso3'].ne('')]
               .groupby(['iso3', 'longitude', 'latitude', 'category'])['total_approved_chf']
               .sum().reset_index())
        cw = (c.pivot_table(index=['iso3', 'longitude', 'latitude'], columns='category',
                            values='total_approved_chf', aggfunc='sum', fill_value=0)
               .reset_index())
        cw.columns.name = None
        for cat in CATS_ORDER:
            if cat not in cw.columns:
                cw[cat] = 0
        cw['total_chf'] = cw[CATS_ORDER].sum(axis=1)

        # Region pivot
        r = (sub.groupby(['region', 'category'])['total_approved_chf']
               .sum().reset_index())
        rw = (r.pivot_table(index='region', columns='category',
                            values='total_approved_chf', aggfunc='sum', fill_value=0)
               .reset_index())
        rw.columns.name = None
        for cat in CATS_ORDER:
            if cat not in rw.columns:
                rw[cat] = 0
        rw['total_chf'] = rw[CATS_ORDER].sum(axis=1)
        rw['longitude'] = rw['region'].map(lambda r: REGION_CENTROIDS.get(r, {}).get('longitude'))
        rw['latitude']  = rw['region'].map(lambda r: REGION_CENTROIDS.get(r, {}).get('latitude'))
        denom = rw['total_chf'].replace(0, np.nan)
        for cat in CATS_ORDER:
            rw[f'{cat}_frac'] = (rw[cat] / denom).fillna(0)

        quarterly.append({'cw': cw, 'rw': rw, 'label': f"{year} Q{q}"})
        if not cw.empty:
            max_country_chf = max(max_country_chf, cw['total_chf'].max())
        if not rw.empty:
            max_region_chf  = max(max_region_chf,  rw['total_chf'].max())

    max_country_chf = max_country_chf or 1.0
    max_region_chf  = max_region_chf  or 1.0

    # ── Compact CHF label helper ──────────────────────────────────────────────
    def _fmt(v):
        if v == 0: return '0'
        if v >= 1e6:
            s = v / 1e6
            return f'{s:.0f}M' if s == int(s) else f'{s:.1f}M'
        if v >= 1e3:
            s = v / 1e3
            return f'{s:.0f}k' if s == int(s) else f'{s:.1f}k'
        return f'{v:.0f}'

    cb_vals  = [max_country_chf * i / 5 for i in range(6)]
    cb_texts = [_fmt(v) for v in cb_vals]

    # Consistent iso3 list across all frames (union of all quarters)
    all_iso3 = sorted(country_wide['iso3'].dropna().unique())

    def _choropleth(cw):
        iso3_to_z = dict(zip(cw['iso3'], cw['total_chf']))
        return go.Choroplethmapbox(
            geojson=countries_geojson,
            featureidkey=f'properties.{GEOJSON_ISO3_KEY}',
            locations=all_iso3,
            z=[iso3_to_z.get(c, 0) for c in all_iso3],
            colorscale=[[0.0, '#F5E6D3'], [1.0, '#7A2D0A']],
            zmin=0, zmax=max_country_chf,
            marker_opacity=0.7, marker_line_width=0.4, marker_line_color='#ffffff',
            colorbar=dict(
                title=dict(text='Country Funding (CHF)', side='top'),
                orientation='h', x=0.5, y=-0.08, xanchor='center', yanchor='top',
                thickness=14, len=0.6, tickvals=cb_vals, ticktext=cb_texts,
            ),
            showscale=True, name='Country CHF',
            hovertemplate='<b>%{location}</b><br>CHF %{z:,.0f}<extra></extra>',
        )

    def _circle_traces(rw):
        traces = []
        for region in REGIONS_ORDER:
            lon_c = REGION_CENTROIDS.get(region, {}).get('longitude', 0)
            lat_c = REGION_CENTROIDS.get(region, {}).get('latitude',  0)
            row   = rw[rw['region'] == region]
            if row.empty or row.iloc[0]['total_chf'] <= 0:
                continue
            r     = row.iloc[0]
            total = r['total_chf']
            r_lat = max(max_radius_deg * math.sqrt(total / max_region_chf), min_radius_deg)
            cos_φ = math.cos(math.radians(lat_c))
            r_lon = r_lat / cos_φ if abs(cos_φ) > 1e-6 else r_lat
            start = -90.0
            for cat in CATS_ORDER:
                color = CATEGORY_COLORS[cat]
                frac  = r[f'{cat}_frac']
                if frac <= 0:
                    continue
                sweep = frac * 360.0
                n_arc = max(2, int(round(sector_points * frac)))
                angs  = [math.radians(start + i * sweep / (n_arc - 1)) for i in range(n_arc)]
                lons  = [lon_c] + [lon_c + r_lon * math.cos(a) for a in angs] + [lon_c]
                lats  = [lat_c] + [lat_c + r_lat * math.sin(a) for a in angs] + [lat_c]
                traces.append(go.Scattermapbox(
                    lon=lons, lat=lats,
                    mode='lines', fill='toself', fillcolor=color,
                    line=dict(color=color, width=0.5),
                    opacity=0.85, hoverinfo='skip', showlegend=False, name=cat,
                ))
                start += sweep
        return traces

    def _hover(rw):
        lons, lats, texts = [], [], []
        for region in REGIONS_ORDER:
            lon_c = REGION_CENTROIDS.get(region, {}).get('longitude', 0)
            lat_c = REGION_CENTROIDS.get(region, {}).get('latitude',  0)
            lons.append(lon_c); lats.append(lat_c)
            row = rw[rw['region'] == region]
            if row.empty or row.iloc[0]['total_chf'] <= 0:
                texts.append(f'<b>{region}</b><br>No data this quarter')
            else:
                r = row.iloc[0]
                texts.append(
                    f"<b>{r['region']}</b><br>"
                    f"Total: CHF {r['total_chf']:,.0f}<br>"
                    f"DREF: {r['DREF_frac']:.0%}  "
                    f"EA: {r['EA_frac']:.0%}  "
                    f"EAP: {r['EAP_frac']:.0%}"
                )
        return go.Scattermapbox(
            lon=lons, lat=lats, mode='markers',
            marker=dict(size=10, color='rgba(0,0,0,0)'),
            text=texts, hoverinfo='text', showlegend=False, name='',
        )

    def _build_frame_figure(qd):
        fig = go.Figure(
            data=[
                _choropleth(qd['cw']),
                *_circle_traces(qd['rw']),
                _hover(qd['rw']),
                *[
                    go.Scattermapbox(
                        lon=[None], lat=[None], mode='markers',
                        marker=dict(size=12, color=color, symbol='circle'),
                        name=cat, showlegend=True,
                    )
                    for cat, color in CATEGORY_COLORS.items()
                ],
            ]
        )

        quarter_total = qd['rw']['total_chf'].sum() if not qd['rw'].empty else 0
        fig.update_layout(
            mapbox=dict(
                accesstoken=MAPBOX_TOKEN,
                style=MAPBOX_STYLE,
                center=dict(lon=10, lat=15),
                zoom=1.35,
            ),
            title=dict(
                text=(
                    f"<b>DREF · World Funding per Quarter</b>  |  {qd['label']}<br>"
                    f"<sup>Total this quarter: CHF {quarter_total:,.0f}  ·  "
                    f"Choropleth = country CHF  ·  Circles = region totals</sup>"
                ),
                x=0.5,
                font=dict(size=15),
            ),
            legend=dict(
                title=dict(text='Appeal Category'), orientation='v',
                x=0.01, y=0.99,
                bgcolor='rgba(255,255,255,0.85)', bordercolor='#cccccc', borderwidth=1,
            ),
            height=height, width=width,
            margin=dict(l=0, r=0, t=65, b=70),
        )
        return fig

    print(f"Prepared {len(quarterly)} quarterly frames for GIF export")
    return quarterly, _build_frame_figure


def save_world_animation_gif(
    output_path=None,
    width=1000,
    height=620,
    frame_duration_ms=850,
    loop=0,
    scale=2,
    max_radius_deg=14.0,
    min_radius_deg=2.5,
    sector_points=24,
    downsample_to=None,
    palette_colors=256,
    dither=Image.Dither.FLOYDSTEINBERG,
    optimize=False,
):
    """
    Render quarterly world funding maps to a GIF without creating a large HTML file.

    Quality notes:
    - `scale=2` renders each frame at 2x internal resolution for sharper labels and borders.
    - `downsample_to=(w, h)` optionally resizes high-res frames with LANCZOS before GIF encoding.
    - GIF is still limited to 256 colours, so very subtle gradients will remain less smooth than PNG.
    """
    quarterly, build_frame_figure = build_animated_world_map(
        width=width,
        height=height,
        max_radius_deg=max_radius_deg,
        min_radius_deg=min_radius_deg,
        sector_points=sector_points,
    )

    frames = []
    for quarter_data in quarterly:
        fig = build_frame_figure(quarter_data)
        png_bytes = fig.to_image(format='png', width=width, height=height, scale=scale)

        frame = Image.open(io.BytesIO(png_bytes)).convert('RGBA')
        if downsample_to is not None:
            frame = frame.resize(downsample_to, Image.Resampling.LANCZOS)

        frame = frame.convert(
            'P',
            palette=Image.Palette.ADAPTIVE,
            colors=palette_colors,
            dither=dither,
        )
        frames.append(frame)
        print(f"Rendered GIF frame: {quarter_data['label']}")

    if output_path is None:
        output_path = OUTPUTS_DIR / 'map_world_animated.gif'

    frames[0].save(
        output_path,
        save_all=True,
        append_images=frames[1:],
        duration=frame_duration_ms,
        loop=loop,
        optimize=optimize,
        disposal=2,
    )
    print(f"Saved → {output_path}")
    return output_path




In [ ]:
gif_path = save_world_animation_gif(
    width=1200,
    height=760,
    scale=3,
    downsample_to=(1200, 760),
    frame_duration_ms=700,
    sector_points=20,
    optimize=False,
)
gif_path